In [1]:
import sys
import json

from pathlib import Path

import joblib
import pandas as pd



# Project Paths

PROJECT_ROOT = (
    Path.cwd()
    .resolve()
    .parent
)


SRC_DIR = (
    PROJECT_ROOT
    / "src"
)


if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

# add original folder into PYTHONPATH
project_root = Path().resolve().parent   
sys.path.append(str(project_root)) 
# Project Services

from ml.churn_model.services.prediction_service import (
    ChurnPredictionService,
)


# Production Artifacts


MODEL_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
    / "churn_pipeline.joblib"
)


METADATA_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
    / "churn_pipeline_metadata.json"
)


print(
    "PROJECT_ROOT:",
    PROJECT_ROOT,
)


print(
    "MODEL_PATH:",
    MODEL_PATH,
)


print(
    "MODEL EXISTS:",
    MODEL_PATH.exists(),
)


print(
    "METADATA PATH:",
    METADATA_PATH,
)


print(
    "METADATA EXISTS:",
    METADATA_PATH.exists(),
)


PROJECT_ROOT: D:\Research_Project\customer-churn-prediction
MODEL_PATH: D:\Research_Project\customer-churn-prediction\artifacts\models\churn_pipeline.joblib
MODEL EXISTS: True
METADATA PATH: D:\Research_Project\customer-churn-prediction\artifacts\models\churn_pipeline_metadata.json
METADATA EXISTS: True


In [2]:
# Load Production Pipeline
if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"Production pipeline not found: "
        f"{MODEL_PATH}"
    )


loaded_model = joblib.load(
    MODEL_PATH
)


print(
    "Loaded object type:",
    type(
        loaded_model
    ).__name__,
)

Loaded object type: Pipeline


In [3]:
# Inspect Pipeline Steps
if hasattr(
    loaded_model,
    "named_steps",
):

    print(
        "Pipeline steps:"
    )

    for name, step in (
        loaded_model
        .named_steps
        .items()
    ):

        print(
            f"{name:22} -> "
            f"{type(step).__name__}"
        )

else:

    raise TypeError(
        "The loaded artifact is not an "
        "end-to-end sklearn Pipeline."
    )

Pipeline steps:
cleaning               -> FunctionTransformer
feature_engineering    -> FunctionTransformer
preprocessor           -> ColumnTransformer
classifier             -> RandomForestClassifier


In [4]:
# Validate Required Pipeline Steps
required_steps = {
    "cleaning",
    "feature_engineering",
    "preprocessor",
    "classifier",
}


actual_steps = set(
    loaded_model
    .named_steps
    .keys()
)


missing_steps = (
    required_steps
    - actual_steps
)


if missing_steps:

    raise ValueError(
        "Invalid end-to-end pipeline. "
        f"Missing steps: "
        f"{missing_steps}"
    )


print(
    "End-to-end pipeline "
    "validated successfully."
)

End-to-end pipeline validated successfully.


In [5]:
# Load Existing Production Metadata
if not METADATA_PATH.exists():

    raise FileNotFoundError(
        "Pipeline metadata not found: "
        f"{METADATA_PATH}"
    )


with METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    metadata = json.load(
        file
    )


print(
    "Production metadata:"
)


print(
    json.dumps(
        metadata,
        indent=4,
    )
)

Production metadata:
{
    "model_name": "End-to-End Tuned Random Forest",
    "model_version": "1.0.0",
    "algorithm": "RandomForestClassifier",
    "task": "Binary Classification",
    "target": "Churn",
    "positive_class": "Churn",
    "negative_class": "No Churn",
    "decision_threshold": 0.47999999999999987,
    "model_selection_metric": "ROC-AUC",
    "threshold_selection_metric": "F1",
    "test_size": 0.2,
    "random_state": 42,
    "features_source": "IBM Telco Customer Churn",
    "status": "Production",
    "artifacts": {
        "pipeline": "churn_pipeline.joblib",
        "metrics": "end_to_end_pipeline_metrics.csv",
        "confusion_matrix": "confusion_matrix.json",
        "roc_curve": "roc_curve.json"
    },
    "metrics": {
        "accuracy": 0.7565649396735273,
        "precision": 0.5269565217391304,
        "recall": 0.8101604278074866,
        "f1": 0.6385669125395153,
        "roc_auc": 0.8420083184789067,
        "pr_auc": 0.6519788988185089
    }
}


In [6]:
import math

# Initialize Prediction Service
prediction_service = (
    ChurnPredictionService(
        model_path=MODEL_PATH,
        metadata_path=METADATA_PATH,
    )
)


print(
    "Prediction service initialized."
)


print(
    "Service threshold:",
    prediction_service.threshold,
)


metadata_threshold = (
    metadata.get(
        "decision_threshold",
        metadata.get(
            "threshold"
        ),
    )
)


print(
    "Metadata threshold:",
    metadata_threshold,
)


if metadata_threshold is not None:

    assert (
        float(
            prediction_service.threshold
        )
        ==
        float(
            metadata_threshold
        )
    )


print(
    "Threshold validation passed."
)

Prediction service initialized.
Service threshold: 0.47999999999999987
Metadata threshold: 0.47999999999999987
Threshold validation passed.


In [7]:

# High-risk sample customer
customer = {

    "gender": "Female",

    "SeniorCitizen": 0,

    "Partner": "No",

    "Dependents": "No",

    "tenure": 5,

    "PhoneService": "Yes",

    "MultipleLines": "No",

    "InternetService":
        "Fiber optic",

    "OnlineSecurity": "No",

    "OnlineBackup": "No",

    "DeviceProtection": "No",

    "TechSupport": "No",

    "StreamingTV": "Yes",

    "StreamingMovies": "Yes",

    "Contract":
        "Month-to-month",

    "PaperlessBilling": "Yes",

    "PaymentMethod":
        "Electronic check",

    "MonthlyCharges": 95.50,

    "TotalCharges": 477.50,
}


result = (
    prediction_service
    .predict(
        customer
    )
)


result

{'prediction': 1,
 'prediction_label': 'Churn',
 'churn_probability': 0.8923,
 'churn_probability_percent': 89.23,
 'threshold': 0.47999999999999987,
 'risk_level': 'Critical',
 'recommendation': 'Immediate retention intervention is recommended. Contact the customer and investigate potential service or pricing concerns.'}

In [8]:
# Verify High-Risk Prediction Result
print(
    "Prediction class:",
    result.get(
        "prediction"
    ),
)


print(
    "Prediction label:",
    result.get(
        "prediction_label"
    ),
)


print(
    "Churn probability:",
    result.get(
        "churn_probability"
    ),
)


print(
    "Churn probability (%):",
    result.get(
        "churn_probability_percent"
    ),
)


print(
    "Decision threshold:",
    prediction_service.threshold,
)


print(
    "Risk level:",
    result.get(
        "risk_level"
    ),
)


print(
    "Recommendation:",
    result.get(
        "recommendation"
    ),
)

Prediction class: 1
Prediction label: Churn
Churn probability: 0.8923
Churn probability (%): 89.23
Decision threshold: 0.47999999999999987
Risk level: Critical
Recommendation: Immediate retention intervention is recommended. Contact the customer and investigate potential service or pricing concerns.


In [9]:
# Low-Risk Sample Customer

low_risk_customer = {

    "gender": "Male",

    "SeniorCitizen": 0,

    "Partner": "Yes",

    "Dependents": "Yes",

    "tenure": 65,

    "PhoneService": "Yes",

    "MultipleLines": "Yes",

    "InternetService": "DSL",

    "OnlineSecurity": "Yes",

    "OnlineBackup": "Yes",

    "DeviceProtection": "Yes",

    "TechSupport": "Yes",

    "StreamingTV": "No",

    "StreamingMovies": "No",

    "Contract": "Two year",

    "PaperlessBilling": "No",

    "PaymentMethod":
        "Bank transfer (automatic)",

    "MonthlyCharges": 65.00,

    "TotalCharges": 4225.00,
}


low_risk_result = (
    prediction_service
    .predict(
        low_risk_customer
    )
)


low_risk_result

{'prediction': 0,
 'prediction_label': 'No Churn',
 'churn_probability': 0.0433,
 'churn_probability_percent': 4.33,
 'threshold': 0.47999999999999987,
 'risk_level': 'Low',
 'recommendation': 'No immediate retention action is required. Continue standard customer engagement.'}

In [10]:
# Compare High-Risk and Low-Risk Predictions

comparison_results = pd.DataFrame(
    [
        {
            "Customer":
                "High Risk",

            "Prediction":
                result.get(
                    "prediction_label"
                ),

            "Probability":
                result.get(
                    "churn_probability"
                ),

            "ProbabilityPercent":
                result.get(
                    "churn_probability_percent"
                ),

            "RiskLevel":
                result.get(
                    "risk_level"
                ),
        },

        {
            "Customer":
                "Low Risk",

            "Prediction":
                low_risk_result.get(
                    "prediction_label"
                ),

            "Probability":
                low_risk_result.get(
                    "churn_probability"
                ),

            "ProbabilityPercent":
                low_risk_result.get(
                    "churn_probability_percent"
                ),

            "RiskLevel":
                low_risk_result.get(
                    "risk_level"
                ),
        },
    ]
)


comparison_results

,Customer,Prediction,Probability,ProbabilityPercent,RiskLevel
0,High Risk,Churn,0.8923,89.23,Critical
1,Low Risk,No Churn,0.0433,4.33,Low


In [11]:
# Batch Prediction
batch_customers = [
    customer,
    low_risk_customer,
]


batch_results = (
    prediction_service
    .predict_batch(
        batch_customers
    )
)


batch_results

[{'prediction': 1,
  'prediction_label': 'Churn',
  'churn_probability': 0.8923,
  'churn_probability_percent': 89.23,
  'threshold': 0.47999999999999987,
  'risk_level': 'Critical',
  'recommendation': 'Immediate retention intervention is recommended. Contact the customer and investigate potential service or pricing concerns.'},
 {'prediction': 0,
  'prediction_label': 'No Churn',
  'churn_probability': 0.0433,
  'churn_probability_percent': 4.33,
  'threshold': 0.47999999999999987,
  'risk_level': 'Low',
  'recommendation': 'No immediate retention action is required. Continue standard customer engagement.'}]

In [12]:
# Batch Prediction Table
batch_results_df = (
    pd.DataFrame(
        batch_results
    )
)


batch_results_df

,prediction,prediction_label,churn_probability,churn_probability_percent,threshold,risk_level,recommendation
0,1,Churn,0.8923,89.23,0.48,Critical,Immediate retention intervention is recommende...
1,0,No Churn,0.0433,4.33,0.48,Low,No immediate retention action is required. Con...


In [13]:
# Validation Test — Empty Customer
try:

    prediction_service.predict(
        {}
    )

except ValueError as error:

    print(
        "Expected validation error:"
    )

    print(
        error
    )

Expected validation error:
Customer data cannot be empty.


In [14]:
# Validation Test — Missing Required Field
invalid_customer = (
    customer.copy()
)


invalid_customer.pop(
    "Contract"
)


try:

    prediction_service.predict(
        invalid_customer
    )

except ValueError as error:

    print(
        "Expected validation error:"
    )

    print(
        error
    )

Expected validation error:
Missing required model fields: Contract


In [15]:
# Robustness Test — Unknown Category
unknown_customer = (
    customer.copy()
)


unknown_customer[
    "PaymentMethod"
] = "Crypto Payment"


try:

    unknown_result = (
        prediction_service
        .predict(
            unknown_customer
        )
    )

    print(
        unknown_result
    )

except Exception as error:

    print(
        type(
            error
        ).__name__,
        ":",
        error,
    )

{'prediction': 1, 'prediction_label': 'Churn', 'churn_probability': 0.8905, 'churn_probability_percent': 89.05, 'threshold': 0.47999999999999987, 'risk_level': 'Critical', 'recommendation': 'Immediate retention intervention is recommended. Contact the customer and investigate potential service or pricing concerns.'}


In [16]:
# Robustness Test — Missing TotalCharges
missing_value_customer = (
    customer.copy()
)


missing_value_customer[
    "TotalCharges"
] = None


missing_value_result = (
    prediction_service
    .predict(
        missing_value_customer
    )
)


missing_value_result

{'prediction': 1,
 'prediction_label': 'Churn',
 'churn_probability': 0.8279,
 'churn_probability_percent': 82.79,
 'threshold': 0.47999999999999987,
 'risk_level': 'Critical',
 'recommendation': 'Immediate retention intervention is recommended. Contact the customer and investigate potential service or pricing concerns.'}

In [17]:
# Model Information
model_info = (
    prediction_service
    .get_model_info()
)


print(
    json.dumps(
        model_info,
        indent=4,
        default=str,
    )
)

{
    "model_name": "End-to-End Tuned Random Forest",
    "model_version": "1.0.0",
    "algorithm": "RandomForestClassifier",
    "task": "Binary Classification",
    "target": "Churn",
    "positive_class": "Churn",
    "negative_class": "No Churn",
    "decision_threshold": 0.47999999999999987,
    "model_selection_metric": "ROC-AUC",
    "threshold_selection_metric": "F1",
    "test_size": 0.2,
    "random_state": 42,
    "features_source": "IBM Telco Customer Churn",
    "status": "Production",
    "artifacts": {
        "pipeline": "churn_pipeline.joblib",
        "metrics": "end_to_end_pipeline_metrics.csv",
        "confusion_matrix": "confusion_matrix.json",
        "roc_curve": "roc_curve.json"
    },
    "metrics": {
        "accuracy": 0.7565649396735273,
        "precision": 0.5269565217391304,
        "recall": 0.8101604278074866,
        "f1": 0.6385669125395153,
        "roc_auc": 0.8420083184789067,
        "pr_auc": 0.6519788988185089
    },
    "threshold": 0.4799999

In [18]:
# Final Prediction Service Verification
print(
    "Production model:"
)

print(
    MODEL_PATH
)


print(
    "\nProduction metadata:"
)

print(
    METADATA_PATH
)


print(
    "\nPipeline classifier:"
)

print(
    type(
        loaded_model
        .named_steps[
            "classifier"
        ]
    ).__name__
)


print(
    "\nDecision threshold:"
)

print(
    prediction_service.threshold
)


print(
    "\nHigh-risk customer:"
)

print(
    result.get(
        "prediction_label"
    ),
    "| probability =",
    result.get(
        "churn_probability"
    ),
)


print(
    "\nLow-risk customer:"
)

print(
    low_risk_result.get(
        "prediction_label"
    ),
    "| probability =",
    low_risk_result.get(
        "churn_probability"
    ),
)


print(
    "\nModel info:"
)

print(
    json.dumps(
        model_info,
        indent=4,
        default=str,
    )
)


print(
    "\nPrediction service verification completed successfully."
)

Production model:
D:\Research_Project\customer-churn-prediction\artifacts\models\churn_pipeline.joblib

Production metadata:
D:\Research_Project\customer-churn-prediction\artifacts\models\churn_pipeline_metadata.json

Pipeline classifier:
RandomForestClassifier

Decision threshold:
0.47999999999999987

High-risk customer:
Churn | probability = 0.8923

Low-risk customer:
No Churn | probability = 0.0433

Model info:
{
    "model_name": "End-to-End Tuned Random Forest",
    "model_version": "1.0.0",
    "algorithm": "RandomForestClassifier",
    "task": "Binary Classification",
    "target": "Churn",
    "positive_class": "Churn",
    "negative_class": "No Churn",
    "decision_threshold": 0.47999999999999987,
    "model_selection_metric": "ROC-AUC",
    "threshold_selection_metric": "F1",
    "test_size": 0.2,
    "random_state": 42,
    "features_source": "IBM Telco Customer Churn",
    "status": "Production",
    "artifacts": {
        "pipeline": "churn_pipeline.joblib",
        "metr